# Aula 7 — Revisão da Aula 6: exercícios resolvidos

**Fundamentos de Programação para Inteligência Artificial na Saúde**
Mestrado e Doutorado Stricto Sensu em Ciências da Saúde — Einstein

Quatro exercícios curtos para retomar a Aula 6 (*Pandas III e NumPy: operações vetorizadas, variáveis derivadas e agregações*) antes de começar a análise exploratória.



In [ ]:
import pandas as pd
import numpy as np

---
## Exercício 1 — Operação vetorizada

Considere a seguinte coluna:

In [ ]:
df_rev = pd.DataFrame({
    "tempo_internacao_dias": [3, 10, None, 5]
})
df_rev

,tempo_internacao_dias
0,3.0
1,10.0
2,NaN
3,5.0


**Enunciado.** Crie `tempo_internacao_semanas` **sem laço** e confira com `np.allclose()` contra a versão com `for`.
Por que `equal_nan=True` é necessário aqui?

### Resolução

**Passo 1 — versão com laço.** Percorremos a coluna valor a valor, como na Aula 3.

In [ ]:
semanas_loop = []
for dias in df_rev["tempo_internacao_dias"]:
    semanas_loop.append(dias / 7)

df_rev["semanas_loop"] = semanas_loop

**Passo 2 — versão vetorizada.** A operação é aplicada à coluna inteira de uma vez.

In [ ]:
df_rev["tempo_internacao_semanas"] = df_rev["tempo_internacao_dias"] / 7
df_rev

,tempo_internacao_dias,semanas_loop,tempo_internacao_semanas
0,3.0,0.428571,0.428571
1,10.0,1.428571,1.428571
2,NaN,NaN,NaN
3,5.0,0.714286,0.714286


**Passo 3 — conferir a troca.** Comparamos as duas versões com e sem `equal_nan=True`.

In [ ]:
print("Sem equal_nan:", np.allclose(df_rev["semanas_loop"], df_rev["tempo_internacao_semanas"]))
print("Com equal_nan:", np.allclose(df_rev["semanas_loop"], df_rev["tempo_internacao_semanas"], equal_nan=True))

Sem equal_nan: False
Com equal_nan: True


In [ ]:
# Por quê? NaN nunca é igual a NaN
np.nan == np.nan

False

---
## Exercício 2 — Duração entre datas

Considere as seguintes colunas:

In [ ]:
df_rev = pd.DataFrame({
    "data_admissao": ["2026-04-01", "2026-04-03", "2026-04-10"],
    "data_alta":     ["2026-04-05", "2026-04-03", "2026-04-08"]
})
df_rev

,data_admissao,data_alta
0,2026-04-01,2026-04-05
1,2026-04-03,2026-04-03
2,2026-04-10,2026-04-08


**Enunciado.** Converta as duas colunas com `pd.to_datetime()`, calcule `tempo_calculado` com `.dt.days` e identifique a linha que precisa ser investigada.

### Resolução

**Passo 1 — converter e conferir os tipos.** Sem a conversão, as colunas são texto e a subtração falha.

In [ ]:
df_rev["data_admissao"] = pd.to_datetime(df_rev["data_admissao"])
df_rev["data_alta"] = pd.to_datetime(df_rev["data_alta"])
df_rev.dtypes

,0
data_admissao,datetime64[ns]
data_alta,datetime64[ns]


**Passo 2 — calcular a duração.** A subtração de duas datas gera uma duração; `.dt.days` extrai o número de dias.

In [ ]:
df_rev["tempo_calculado"] = (df_rev["data_alta"] - df_rev["data_admissao"]).dt.days
df_rev

,data_admissao,data_alta,tempo_calculado
0,2026-04-01,2026-04-05,4
1,2026-04-03,2026-04-03,0
2,2026-04-10,2026-04-08,-2


**Passo 3 — localizar a linha suspeita.**

In [ ]:
df_rev.loc[df_rev["tempo_calculado"] < 0]

,data_admissao,data_alta,tempo_calculado
2,2026-04-10,2026-04-08,-2


**Interpretação.**
- **4 dias**: valor esperado.
- **0 dias**: plausível — alta no mesmo dia da admissão.
- **−2 dias**: a alta aparece **antes** da admissão. É uma inconsistência a investigar (erro de digitação, troca de colunas, outra definição de data), **não** a corrigir automaticamente.

---
## Exercício 3 — Criando categorias

Considere a seguinte coluna:

In [ ]:
df_rev = pd.DataFrame({
    "idade": [17, 45, 68, 82, None]
})
df_rev

,idade
0,17.0
1,45.0
2,68.0
3,82.0
4,NaN


**Enunciado.** Use `np.select()` para criar `faixa_etaria` com `"menor de 18"`, `"18 a 59"` e `"60 ou mais"`.
Que categoria recebe a idade ausente?

### Resolução

**Passo 1 — listar condições e categorias.** As condições são avaliadas em ordem; a primeira verdadeira define a categoria.

In [ ]:
condicoes = [
    df_rev["idade"] < 18,
    df_rev["idade"].between(18, 59),
    df_rev["idade"] >= 60
]
categorias = ["menor de 18", "18 a 59", "60 ou mais"]

**Passo 2 — aplicar `np.select()` com um `default` explícito.**

In [ ]:
df_rev["faixa_etaria"] = np.select(condicoes, categorias, default="nao classificado")
df_rev

,idade,faixa_etaria
0,17.0,menor de 18
1,45.0,18 a 59
2,68.0,60 ou mais
3,82.0,60 ou mais
4,NaN,nao classificado


**Passo 3 — por que a ausência caiu no `default`?**

In [ ]:
pd.DataFrame({
    "idade": df_rev["idade"],
    "< 18": condicoes[0],
    "18 a 59": condicoes[1],
    ">= 60": condicoes[2]
})

,idade,< 18,18 a 59,>= 60
0,17.0,True,False,False
1,45.0,False,True,False
2,68.0,False,False,True
3,82.0,False,False,True
4,NaN,False,False,False


**Interpretação.** Toda comparação com `NaN` resulta em `False`. Como nenhuma condição é verdadeira na última linha, ela recebe o valor de `default`: `"nao classificado"`.
Por isso vale sempre definir um `default` que **mostre** o que ficou sem classificação, em vez de esconder o problema.

> Gancho para hoje: `faixa_etaria` aparece na Aula 7 como uma variável **categórica**.

---
## Exercício 4 — Resumindo grupos

Considere a seguinte base:

In [ ]:
df_rev = pd.DataFrame({
    "servico": ["Cardio", "Cardio", "Neuro", "Neuro", "Neuro"],
    "tempo_internacao_dias": [4, None, 6, 8, 10],
    "readmissao_30d": [0, 1, 0, 0, 1]
})
df_rev

,servico,tempo_internacao_dias,readmissao_30d
0,Cardio,4.0,0
1,Cardio,NaN,1
2,Neuro,6.0,0
3,Neuro,8.0,0
4,Neuro,10.0,1


**Enunciado.** Com `groupby()` e `agg()`, obtenha por serviço: `n_pacientes` (`size`), `n_tempo` (`count`), `tempo_medio` e `readmissao_prop`.
Por que `n_pacientes` e `n_tempo` diferem em Cardio?

### Resolução

Cada indicador é escrito como `nome=(coluna, função)`, o que deixa explícito como foi calculado.

In [ ]:
indicadores = df_rev.groupby("servico").agg(
    n_pacientes=("tempo_internacao_dias", "size"),
    n_tempo=("tempo_internacao_dias", "count"),
    tempo_medio=("tempo_internacao_dias", "mean"),
    readmissao_prop=("readmissao_30d", "mean")
).reset_index()

indicadores.round(2)

,servico,n_pacientes,n_tempo,tempo_medio,readmissao_prop
0,Cardio,2,1,4.0,0.50
1,Neuro,3,3,8.0,0.33


**Interpretação.**
- `size()` conta **todas as linhas** do grupo; `count()` conta só os **valores não ausentes** da coluna.
- Em Cardio há 2 pacientes, mas apenas 1 tem `tempo_internacao_dias` registrado. Assim, `tempo_medio = 4.0` foi calculado sobre **um único paciente**.
- `readmissao_prop` é a média de uma variável 0/1, ou seja, a **proporção** de readmissões: 0,50 em Cardio e 0,33 em Neuro. É um indicador descritivo, não causal.

> Ponte para a Aula 7: uma média sozinha não mostra quantos valores a sustentam nem como eles se distribuem. É isso que vamos examinar hoje.